In [24]:
import pandas as pd

# Load dataset
data = pd.read_csv('uber.csv')
data.head()  


,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [25]:
# Check for null values in each column
null_values = data.isnull().sum()
print("Null values in each column:")
print(null_values)


Null values in each column:
Unnamed: 0           0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64


In [26]:
# Check for duplicate rows
duplicate_rows = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")


Number of duplicate rows: 0


In [27]:
# Drop the Unnamed: 0 column if it exists
data = data.drop(columns=['Unnamed: 0'])

data.dropna(inplace=True)


In [31]:


# Ensure the column name is consistent
if 'pickup date time' in data.columns:
    data.rename(columns={'pickup date time': 'pickup_datetime'}, inplace=True)
elif 'pickup_datetime ' in data.columns:  # Extra space case
    data.rename(columns={'pickup_datetime ': 'pickup_datetime'}, inplace=True)


# Convert pickup_datetime to datetime
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')

# Extract time-based features
data['pickup_hour'] = data['pickup_datetime'].dt.hour
data['pickup_day'] = data['pickup_datetime'].dt.day
data['pickup_month'] = data['pickup_datetime'].dt.month
data['pickup_dayofweek'] = data['pickup_datetime'].dt.dayofweek

# Drop original pickup_datetime column
data = data.drop(columns=['pickup_datetime'])


# Calculate and print correlation matrix
correlation_matrix = data.corr()
print(correlation_matrix['fare_amount'])


KeyError: 'pickup_datetime'

In [6]:
# Convert pickup_datetime to datetime format
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')

# Extract hour and day of the week
data['hour'] = data['pickup_datetime'].dt.hour
data['day_of_week'] = data['pickup_datetime'].dt.dayofweek


In [8]:
# Calculate Distance Using the Haversine Formula
from math import radians, cos, sin, asin, sqrt

# Define Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# Apply Haversine function to calculate distance_km
data['distance_km'] = data.apply(
    lambda row: haversine(row['pickup_latitude'], row['pickup_longitude'],
                          row['dropoff_latitude'], row['dropoff_longitude']),
    axis=1
)

In [9]:
# Define feature and target variables
X = data[['distance_km', 'hour', 'day_of_week', 'passenger_count']]
y = data['fare_amount']


In [10]:
print(X.head())
print(y.head())

   distance_km  hour  day_of_week  passenger_count
0     1.683323    19            3                1
1     2.457590    20            4                1
2     5.036377    21            0                1
3     1.661683     8            4                3
4     4.475450    17            3                5
0     7.5
1     7.7
2    12.9
3     5.3
4    16.0
Name: fare_amount, dtype: float64


In [13]:
print(X.isna().sum())


distance_km        1
hour               0
day_of_week        0
passenger_count    0
dtype: int64


In [14]:
# Drop rows with any NaN values in X or y
X = X.dropna()
y = y[X.index]  # Ensure y is aligned with X after dropping rows


In [15]:
# train-test split


from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [16]:
#  train the model

from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

# Train the model on the training data
model.fit(X_train, y_train)


LinearRegression()

In [17]:
#  Make Predictions and Evaluate the Model

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)


Mean Absolute Error (MAE): 6.065427876660941
Root Mean Squared Error (RMSE): 10.19410467323295
